# Corpus de mariposas del Tolima

Inventario del corpus descargado y publicación de la partición que consume el
repositorio de modelado.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from dataSplits import SPLITS, contar, imagenesDe, inventariar, publicar, repartir

## Inventario

In [ ]:
conteos = inventariar()
inventario = pd.Series(conteos, name="imagenes").sort_values()
utilizables = inventario[inventario > 0]

pd.Series({
    "carpetas": len(inventario),
    "utilizables": len(utilizables),
    "vacias": int((inventario == 0).sum()),
    "imagenes": int(inventario.sum()),
    "mediana": float(utilizables.median()),
    "minimo": int(utilizables.min()),
    "maximo": int(utilizables.max()),
}).to_frame("valor")

In [ ]:
inventario[inventario < 40].to_frame("imagenes")

## Balance

La cola izquierda son las especies con poco material disponible bajo licencia
abierta, no las que el mariposario tiene menos.

In [ ]:
figura, ax = plt.subplots(figsize=(8, 0.2 * len(utilizables)))
ax.barh(range(len(utilizables)), utilizables.values, height=0.7)
ax.set_yticks(range(len(utilizables)),
              [e.replace("_", " ") for e in utilizables.index],
              fontsize=7, style="italic")
ax.set(xlabel="Imágenes", title="Balance del corpus por especie")
ax.axvline(40, linestyle="--", linewidth=1)

figura.tight_layout()
Path("figuras").mkdir(exist_ok=True)
figura.savefig("figuras/balance_corpus.png", bbox_inches="tight")
plt.show()

## Muestras de las especies más escasas

La revisión visual es donde se decide si el problema es de cantidad o de calidad.

In [ ]:
from PIL import Image
from dataSplits import CORPUS

def rejilla(especies, porEspecie=4):
    figura, ejes = plt.subplots(len(especies), porEspecie,
                                figsize=(2.2 * porEspecie, 2.3 * len(especies)),
                                squeeze=False)
    for fila, especie in enumerate(especies):
        archivos = imagenesDe(especie)[:porEspecie]
        for columna, ax in enumerate(ejes[fila]):
            ax.set_xticks([]); ax.set_yticks([])
            if columna < len(archivos):
                ax.imshow(Image.open(CORPUS / especie / archivos[columna]).convert("RGB"))
            if columna == 0:
                ax.set_ylabel(especie.replace("_", " "), fontsize=8, rotation=0,
                              ha="right", va="center", labelpad=12, style="italic")
    figura.tight_layout()
    plt.show()

rejilla(list(utilizables.index[:4]))

## Especies del mismo género

Los pares que un clasificador confunde primero.

In [ ]:
generos = {}
for especie in utilizables.index:
    generos.setdefault(especie.split("_")[0], []).append(especie)

confundibles = {g: v for g, v in sorted(generos.items()) if len(v) > 1}
pd.Series({g: ", ".join(v) for g, v in confundibles.items()}).to_frame("especies")

In [ ]:
rejilla(max(confundibles.values(), key=len)[:4])

## Publicar la partición

70/20/10, con la semilla combinada con el nombre de la especie. El archivo
resultante es la frontera con el repositorio de modelado.

In [ ]:
SEMILLA = 42

reparto = repartir(utilizables.index, semilla=SEMILLA)
ruta = publicar(reparto, semilla=SEMILLA)

print(f"{len(reparto)} clases · {contar(reparto)}")
print(f"→ {ruta}")

In [ ]:
for r in sorted(Path("splits").glob("*.json")):
    d = json.loads(r.read_text())
    print(f"{r.name}: {d['num_clases']} clases · {d['conteos']}")